# 01 · 共享 global R1 的真实层 G1

逐层采集 q/up 输入侧和 down 输出侧；所有面板共享同一个 Q。完整 reduction groups 与输出 hidden 轴保留。默认只执行 preflight，不下载或启动模型。

GitHub / VS Code 工作流先运行 `00_Colab_GitHub_Workspace.ipynb`；详见 [操作说明](../docs/COLAB_VSCODE.md)。已有 ZIP 入口仍可用。

## 路径与环境配置

GitHub / VS Code 首先运行 `00_Colab_GitHub_Workspace.ipynb`；本 notebook 会识别 `/content/CellQuant` 并读取运行时路径配置。
已有 ZIP 仍可设置 `KIT_ARCHIVE` 和 `EXTRACT_ARCHIVE=True` 解压，或直接指定 `PROJECT_ROOT`。
默认 smoke / manifest-only，不下载模型、不启动全量 sweep。基础依赖缺失时才设置 `INSTALL_BASE_DEPENDENCIES=True`，不替换宿主 torch。
真实模型授权通过环境变量或隐藏输入提供，不能写入 notebook 输出或 Git；阶段结束用 workspace backup 保存到 Drive。


In [7]:
from pathlib import Path
import os, sys, json

# Colab: Git clone 到 /content/CellQuant；也兼容完整 ZIP 解压目录。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or ("/content/CellQuant" if Path("/content/CellQuant/pyproject.toml").is_file() else None)
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = Path("/content").is_dir()  # Colab 默认挂载，本地/AutoDL 默认跳过
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = True
SEED = 20261115

## 挂载 Google Drive

运行下方单元，按 Google 弹窗完成登录授权，然后继续运行后续单元。
此命令同时用于浏览器 Colab 和 VS Code 的 Colab 内核；已挂载时会复用，非 Colab 环境默认跳过。
`CellQuant` 项目文件夹会在后续初始化中自动创建。


In [8]:
# 在 Colab（浏览器或 VS Code 内核）中直接挂载 Google Drive。
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive 挂载尚未完成，请完成授权后重跑本单元。")
    print("Google Drive 已挂载：/content/drive/MyDrive")
else:
    print("非 Colab 环境或 MOUNT_DRIVE=False：跳过云盘挂载。")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 已挂载：/content/drive/MyDrive


In [9]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache/huggingface"))
os.environ.setdefault("PYTHONUTF8", "1")

# Separate notebook kernels reuse the same local runtime path configuration.
workspace_record = PROJECT_ROOT / ".cache/colab_workspace.json"
if workspace_record.is_file():
    from cellquant.workspace import configure_workspace
    saved = json.loads(workspace_record.read_text(encoding="utf-8"))
    configure_workspace(PROJECT_ROOT, saved["environment"]["CELLQUANT_DRIVE_ROOT"])


Project: /content/CellQuant
Runs: /content/CellQuant/runs


先显式准备：`python scripts/prepare_spinquant.py`；`python scripts/prepare_experiment.py --config configs/colab_pilot.json --output runs/prepared/g1`。设置 HF_TOKEN 环境变量或 Colab Secrets，禁止打印。T4 必须记录 FP16；本实现需要足够 CPU RAM 暂存权重和完整面板，高 RAM Colab 才可能满足。

In [10]:
from cellquant.experiment import run_experiment
from cellquant.runtime import stream_command, read_json
from cellquant.gates import evaluate_file
OUT = PATHS["runs"] / "01_pilot"
OUT.mkdir(parents=True, exist_ok=True)
RUN_REAL_MODEL = True
PILOT_CONFIG_PATH = Path(os.environ.get("CELLQUANT_PILOT_CONFIG", PROJECT_ROOT / "runs/prepared/g1/config.json"))
if not PILOT_CONFIG_PATH.is_absolute():
    PILOT_CONFIG_PATH = PROJECT_ROOT / PILOT_CONFIG_PATH
if not PILOT_CONFIG_PATH.is_file():
    if RUN_REAL_MODEL:
        raise FileNotFoundError(
            f"真实 G1 的准备配置不存在：{PILOT_CONFIG_PATH}\n"
            "请在 00_Colab_GitHub_Workspace.ipynb 设置 PREPARE_G1_INPUTS=True，"
            "运行“准备真实 G1 输入”单元，完成数据准备、权重校验和 Drive 备份后再重跑本单元。\n"
            "HF 登录单元下载的 config.json 只是模型说明文件，不是本实验的准备配置。"
        )
    print("准备配置未找到；仅以草案进行预检查，不执行模型。")
    PILOT_CONFIG_PATH = PROJECT_ROOT / "configs/colab_pilot.json"
print("实际使用的 G1 配置：", PILOT_CONFIG_PATH)
preflight = run_experiment(PILOT_CONFIG_PATH, project_root=PROJECT_ROOT, preflight_only=True)
print(preflight["readiness"])


实际使用的 G1 配置： /content/CellQuant/runs/prepared/g1/config.json
{'status': 'READY', 'missing': [], 'environment': {'schema_version': '1.0', 'created_at_utc': '2026-11-16T11:08:54.581972+00:00', 'python': '3.12.13', 'executable': '/usr/bin/python3', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.35', 'machine': 'x86_64', 'packages': {'numpy': '2.0.2', 'scipy': '1.16.3', 'torch': '2.11.0+cu128', 'transformers': '4.44.2', 'datasets': '2.20.0', 'accelerate': '0.34.2', 'safetensors': '0.8.0', 'nbformat': '5.10.4', 'nbclient': '0.10.4', 'fast-hadamard-transform': '1.1.0', 'flash-attn': None, 'triton': '3.6.0', 'lm_eval': None, 'tokenizers': '0.19.1', 'huggingface-hub': '0.36.2', 'pyarrow': '18.1.0', 'psutil': '5.9.5'}, 'cuda': {'available': True, 'torch_cuda_build': '12.8', 'status': 'DEVICE_DETECTED', 'devices': [{'index': 0, 'name': 'NVIDIA L4', 'total_gib': 22.0343017578125, 'compute_capability': [8, 9]}], 'bf16_supported': True, 'nvidia_smi': 'NVIDIA L4, 580.82.07, 23034 MiB, 8.9'}, 'compat

In [11]:
if RUN_REAL_MODEL:
    if preflight["readiness"]["status"] != "READY":
        reasons = "\n".join("- " + reason for reason in preflight["readiness"]["missing"])
        raise RuntimeError(
            f"真实 G1 预检查仍为 BLOCKED。配置：{PILOT_CONFIG_PATH}\n"
            f"{reasons}\n"
            "先解决这些前置项并重跑上一个预检查单元；RUN_REAL_MODEL 不能代替科学证据。"
        )
    import uuid
    PILOT_LOG = OUT / f"pilot_{uuid.uuid4().hex[:12]}.log"
    print("Pilot log:", PILOT_LOG, flush=True)
    process = stream_command([sys.executable, "-u", "-m", "cellquant.experiment", "--config", str(PILOT_CONFIG_PATH)],
                             PILOT_LOG, cwd=PROJECT_ROOT, timeout_s=5*3600)
    if process["returncode"] != 0:
        raise RuntimeError(f"Real pilot incomplete; inspect {PILOT_LOG}")
else:
    print("NOT_RUN: real-model execution switch is false")


Pilot log: /content/CellQuant/runs/01_pilot/pilot_ea9646eb75d0.log
[CellQuant] run_dir=/content/CellQuant/runs/experiments/2026-11-16T110856.019129_0000_66c2b1eb47 pid=39830
[CellQuant] {"event": "progress", "status": "RUNNING", "phase": "initialization", "details": {}, "elapsed_seconds": 0.0, "phase_elapsed_seconds": 0.0, "process_rss_gib": 0.546, "system_available_gib": 28.428, "process_cpu_seconds": 4.94, "progress_sequence": 1, "updated_at_utc": "2026-11-16T11:08:58.459771+00:00"}
[CellQuant] {"event": "progress", "status": "RUNNING", "phase": "load", "details": {}, "elapsed_seconds": 0.03, "phase_elapsed_seconds": 0.0, "process_rss_gib": 0.546, "system_available_gib": 28.426, "process_cpu_seconds": 4.96, "progress_sequence": 2, "updated_at_utc": "2026-11-16T11:08:58.491723+00:00"}
[CellQuant] {"event": "heartbeat", "status": "RUNNING", "phase": "load", "details": {}, "elapsed_seconds": 30.0, "phase_elapsed_seconds": 29.97, "process_rss_gib": 0.654, "system_available_gib": 28.334, 

KeyboardInterrupt: 

In [ ]:
import psutil, time
from pathlib import Path

assert Path("/content/CellQuant").is_dir(), "请在原 Colab 服务器运行"

for sample in range(2):
    mem = psutil.virtual_memory()
    print(f"\n采样 {sample + 1}：剩余 RAM {mem.available / 2**30:.1f} GiB")
    jobs = [
        p for p in psutil.process_iter(["cmdline"])
        if "cellquant.experiment" in (p.info["cmdline"] or [])
    ]
    print("实验进程数量：", len(jobs))
    for p in jobs:
        try:
            cpu = p.cpu_percent(interval=1)
            t = p.cpu_times()
            print("PID:", p.pid, "CPU%:", cpu,
                  "累计CPU秒:", round(t.user + t.system, 1),
                  "RSS GiB:", round(p.memory_info().rss / 2**30, 1))
        except psutil.Error as e:
            print(p.pid, type(e).__name__)
    if sample == 0:
        time.sleep(10)

### 查看运行进度（只读）

新版实际进程每 30 秒输出资源心跳，阶段变化时输出层号、seed、方法、面板或坐标对；`heartbeat` 只说明进程存活，不表示算法有进展。模型先驻留 CPU，采集逐层使用 GPU，默认求解在 CPU，故 GPU 不会一直繁忙。每个方法的预算也不包含全部准备及 held-out 计算，不能把预算当作完成时间承诺。

如果上方单元仍在执行，同一 kernel 中的下方单元会排队。请在能访问同一个 `PROJECT_ROOT` 的独立终端运行 `python -m cellquant.diagnostics --project-root /content/CellQuant`，或将下方只读单元复制到同一服务器的独立 kernel；不需要另建 Colab 服务器。隔 30 秒检查两次 CPU 累计时间、进度序号、剩余 RAM 与进程状态。旧代码的 `fusion` 状态可能已经过时，诊断会明确提示；单次 CPU 为零不能证明死锁。

不要重复启动模型进程。正在运行的旧进程不会自动得到新版日志；先诊断并保留旧日志，待作业结束或主动停止后再 pull、重启 kernel，以新 run 运行，不能把新代码混接进旧实验。


In [ ]:
from cellquant.diagnostics import inspect_runs
print(json.dumps(inspect_runs(PROJECT_ROOT), ensure_ascii=False, indent=2))


真实作业写入唯一 `runs/experiments/<run_id>`。把该目录中的 G1_evidence.json 设为下方路径；G1 从原始测量重算。运行标志不会更改门槛。超事件/时间预算记录 RESOURCE_BLOCKED；未完成控制组不产生性能结论。

In [ ]:
G1_EVIDENCE = Path(os.environ.get("CELLQUANT_G1_EVIDENCE", PROJECT_ROOT / "runs/gates/G1_evidence.json"))
evidence_path = G1_EVIDENCE if G1_EVIDENCE.is_file() else PROJECT_ROOT / "examples/G1_evidence_NOT_RUN.json"
gate = evaluate_file("G1", evidence_path)
atomic_json(OUT / "G1.json", gate)
print(gate["status"], gate["missing_fields"])


INSUFFICIENT_EVIDENCE ['source_type', 'provenance', 'artifact_refs', 'metrics', 'metrics.g0_pass', 'metrics.real_panel_provenance_verified', 'metrics.real_model_parameter_billions', 'metrics.calibration_seed_count', 'metrics.layer_count', 'metrics.site_count', 'metrics.fit_sequence_count', 'metrics.heldout_sequence_count', 'metrics.calibration_heldout_disjoint', 'metrics.shared_global_r1', 'metrics.full_quant_groups_preserved', 'metrics.down_projection_output_side_correct', 'metrics.matched_solver_controls_complete', 'metrics.same_hardware_timing', 'metrics.each_seed_pooled_heldout_improved', 'metrics.max_site_nmse_regression_vs_best_pct', 'metrics.median_nmse_gain_vs_hadamard_pct', 'metrics.median_nmse_gain_vs_best_matched_solver_pct', 'metrics.site_win_fraction_vs_best_matched_solver', 'metrics.median_nmse_regression_vs_best_matched_solver_pct', 'metrics.site_noninferior_fraction_at_2pct', 'metrics.pilot_calibration_speedup']
